# Final 04: Reddit Held-Out Test Routed Phase 2 Reasoning and End-to-End Evaluation

This notebook is an optional final paper workflow step for the primary Reddit held-out test set.

Run order:

1. Run `01_distilbert_phase1_training_final_colab.ipynb` first.
2. Confirm that `outputs_final/phase1_distilbert/phase1_test_predictions.csv` exists in Google Drive.
3. Run this notebook.

This notebook does not train DistilBERT and does not select a new threshold. It reads the Reddit held-out test predictions produced by Final 01, selects rows where `phase1_routed=True`, applies Llama 2 CoT and Llama 3 SELF-DISCOVER reasoning only to those routed examples, and then reconstructs the full Reddit held-out test end-to-end result.

Use this notebook to compare the best Phase 1 model alone against the full confidence-guided two-phase framework on the primary Reddit held-out test set.



In [ ]:
# Colab setup. Run this cell first in a fresh Colab GPU runtime.
# IMPORTANT: after this cell finishes, restart the runtime once, then run from the imports cell.
%pip install -q -U pandas tqdm scikit-learn sentencepiece protobuf accelerate transformers "bitsandbytes>=0.46.1"

import importlib.metadata as importlib_metadata
print("bitsandbytes:", importlib_metadata.version("bitsandbytes"))
print("transformers:", importlib_metadata.version("transformers"))
print("accelerate:", importlib_metadata.version("accelerate"))

print("\nSETUP COMPLETE. Now restart the runtime once: Runtime > Restart runtime, then rerun from the import cell.")


In [ ]:
import os
import gc
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Image
from tqdm.auto import tqdm

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

try:
    from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support
except Exception:
    accuracy_score = classification_report = confusion_matrix = None

LABELS = ["Depression", "Neutral", "Happy"]





In [ ]:
# Dependency check. Run this after restarting the runtime.
import importlib.metadata as importlib_metadata
import importlib.util

bnb_spec = importlib.util.find_spec("bitsandbytes")
print("bitsandbytes spec:", bnb_spec)
print("bitsandbytes version:", importlib_metadata.version("bitsandbytes"))

if bnb_spec is None:
    raise RuntimeError("bitsandbytes is not importable. Re-run the setup cell, restart runtime, and rerun from the imports cell.")


## Configuration

For final paper results, this notebook should read the Reddit held-out test Phase 1 output produced by Final 01:

`/content/drive/MyDrive/confidence_guided_llm_reasoning/outputs_final/phase1_distilbert/phase1_test_predictions.csv`

The notebook filters to `phase1_routed=True`. Only those low-confidence Reddit test examples are sent to Llama 2 / Llama 3. Accepted Phase 1 examples are not reprocessed; they keep the DistilBERT Phase 1 label in the final end-to-end evaluation.

All outputs are saved to Google Drive under `outputs_final/reddit_test_phase2_reasoning_universal_prompt_v2/`, and each completed Llama row is appended immediately so the run can resume after a Colab disconnect.



## Universal Prompt Policy v2 Experiment

This notebook uses the same final universal-policy-v2 CoT and SELF-DISCOVER prompts as the Mixed Emotion Phase 2 workflow. The prompts are applied only to Reddit held-out test examples that the calibrated DistilBERT Phase 1 model routed because their confidence fell below the selected threshold.



In [ ]:
# Final Phase 1 Reddit held-out test output. This is produced by 01_distilbert_phase1_training_final_colab.ipynb.
PHASE1_TEST_PREDICTIONS_PATH = Path("/content/drive/MyDrive/confidence_guided_llm_reasoning/outputs_final/phase1_distilbert/phase1_test_predictions.csv")

# Save outputs to Google Drive so row-level checkpoints survive runtime resets.
USE_GOOGLE_DRIVE_OUTPUT = True
REQUIRE_PERSISTENT_OUTPUT = True
LOCAL_OUTPUT_DIR = Path("outputs_final/reddit_test_phase2_reasoning_universal_prompt_v2")
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/confidence_guided_llm_reasoning/outputs_final/reddit_test_phase2_reasoning_universal_prompt_v2")

OUTPUT_DIR = LOCAL_OUTPUT_DIR
DRIVE_OUTPUT_AVAILABLE = False

if USE_GOOGLE_DRIVE_OUTPUT:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        probe_path = DRIVE_OUTPUT_DIR / "_drive_write_test.txt"
        probe_path.write_text("ok", encoding="utf-8")
        probe_path.unlink(missing_ok=True)
        OUTPUT_DIR = DRIVE_OUTPUT_DIR
        DRIVE_OUTPUT_AVAILABLE = True
        print(f"Google Drive output enabled: {OUTPUT_DIR}")
    except Exception as exc:
        if REQUIRE_PERSISTENT_OUTPUT:
            raise RuntimeError(
                "Google Drive output is unavailable, so the notebook stopped before running expensive inference. "
                "Fix Drive authorization/mount first, or set REQUIRE_PERSISTENT_OUTPUT = False only for a temporary smoke test."
            ) from exc
        print(f"Google Drive output is unavailable ({exc}); falling back to local runtime output.")
        OUTPUT_DIR = LOCAL_OUTPUT_DIR
else:
    if REQUIRE_PERSISTENT_OUTPUT:
        raise RuntimeError("Turn on Drive output or set REQUIRE_PERSISTENT_OUTPUT=False for temporary smoke tests.")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TEXT_COL = "text"
TRUE_LABEL_COL = "target_label"
PHASE1_LABEL_COL = "phase1_label"
PHASE1_ROUTED_COL = "phase1_routed"
PHASE1_ACCEPTED_COL = "phase1_accepted"

# Final Reddit end-to-end mode: use only rows routed by Phase 1 for LLM reasoning.
RUN_ROUTED_ONLY = True
MAX_ROWS = None  # final run uses all routed Reddit held-out test rows; set small integer for smoke testing.
RANDOM_STATE = 42

RUN_LLAMA2_COT = True
RUN_LLAMA3_SELF_DISCOVER = True

LLAMA2_MODEL_NAME = "NousResearch/Llama-2-7b-chat-hf"
LLAMA3_MODEL_NAME = "NousResearch/Meta-Llama-3-8B-Instruct"
SELF_DISCOVER_STRUCTURE_MODE = "per_sample"

MAX_NEW_TOKENS_COT = 256
MAX_NEW_TOKENS_SELF_DISCOVER = 1024

LLAMA2_OUTPUT_PATH = OUTPUT_DIR / "reddit_test_llama2_cot_routed_results.csv"
LLAMA3_OUTPUT_PATH = OUTPUT_DIR / "reddit_test_llama3_self_discover_routed_results.csv"
RESUME_FROM_EXISTING = True
print(f"Output directory: {OUTPUT_DIR}")



In [ ]:
# Optional Hugging Face token support for gated or rate-limited model access.
# In Colab, add a secret named HF_TOKEN if needed.
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        os.environ["HUGGINGFACE_HUB_TOKEN"] = hf_token
        print("HF_TOKEN loaded from Colab secrets.")
except Exception:
    print("Colab userdata not available or HF_TOKEN not set. Continuing without explicit HF token.")


## Load Reddit Held-Out Test Phase 1 Predictions


In [ ]:
def _normalize_bool(value):
    return str(value).strip().lower() in {"true", "1", "yes", "y"}


def load_phase2_input():
    if not PHASE1_TEST_PREDICTIONS_PATH.exists():
        raise FileNotFoundError(
            f"Missing Reddit held-out Phase 1 predictions: {PHASE1_TEST_PREDICTIONS_PATH}. "
            "Run Final 01 first and confirm that phase1_test_predictions.csv was saved to Google Drive."
        )

    print("Loading Reddit held-out Phase 1 predictions:", PHASE1_TEST_PREDICTIONS_PATH)
    df = pd.read_csv(PHASE1_TEST_PREDICTIONS_PATH)

    if TRUE_LABEL_COL not in df.columns:
        if "label_str" in df.columns:
            df[TRUE_LABEL_COL] = df["label_str"]
        elif "label" in df.columns:
            id_to_label = {0: "Depression", 1: "Neutral", 2: "Happy"}
            df[TRUE_LABEL_COL] = df["label"].map(id_to_label)
        else:
            raise ValueError("Could not derive target_label. Expected target_label, label_str, or label column.")

    required = {"example_id", TEXT_COL, TRUE_LABEL_COL, PHASE1_LABEL_COL, PHASE1_ROUTED_COL}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Phase 1 Reddit test prediction file missing columns: {missing}")

    df["phase1_label_for_prompt"] = df[PHASE1_LABEL_COL]
    df[PHASE1_ROUTED_COL] = df[PHASE1_ROUTED_COL].map(_normalize_bool)
    if PHASE1_ACCEPTED_COL in df.columns:
        df[PHASE1_ACCEPTED_COL] = df[PHASE1_ACCEPTED_COL].map(_normalize_bool)
    else:
        df[PHASE1_ACCEPTED_COL] = ~df[PHASE1_ROUTED_COL]
    df["input_source"] = "reddit_held_out_test_phase1_predictions"
    return df


df = load_phase2_input()
print("All Reddit held-out test rows:", df.shape)
print("Routed rows:", int(df[PHASE1_ROUTED_COL].sum()))
print("Routing rate:", float(df[PHASE1_ROUTED_COL].mean()))
display(df.head())
print(df[TRUE_LABEL_COL].value_counts())



In [ ]:
required = {"example_id", TEXT_COL, TRUE_LABEL_COL, "phase1_label_for_prompt"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

work_df = df.copy()
if RUN_ROUTED_ONLY and PHASE1_ROUTED_COL in work_df.columns:
    work_df = work_df[work_df[PHASE1_ROUTED_COL].astype(str).str.lower().isin(["true", "1", "yes"])]
    print(f"Using routed rows only: {len(work_df)} rows")
elif RUN_ROUTED_ONLY:
    print("RUN_ROUTED_ONLY=True but phase1_routed column is absent. Running all loaded rows.")

if MAX_ROWS is not None:
    work_df = work_df.sample(n=min(MAX_ROWS, len(work_df)), random_state=RANDOM_STATE).reset_index(drop=True)
else:
    work_df = work_df.reset_index(drop=True)

if work_df.empty:
    raise ValueError("No rows selected for Phase 2 reasoning. Check threshold/routing settings.")

print(work_df.shape)
display(work_df[["example_id", TRUE_LABEL_COL, "phase1_label_for_prompt", TEXT_COL]].head())
work_df.to_csv(OUTPUT_DIR / "reddit_test_routed_input_rows.csv", index=False)



## Model Loading Helpers

In [ ]:
def load_chat_model(model_name, load_in_4bit=True):
    compute_dtype = torch.float16
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        trust_remote_code=True,
        token=os.environ.get("HF_TOKEN"),
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    quantization_config = None
    if load_in_4bit:
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=compute_dtype,
            bnb_4bit_use_double_quant=True,
        )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        dtype=compute_dtype,
        device_map="auto",
        quantization_config=quantization_config,
        trust_remote_code=True,
        token=os.environ.get("HF_TOKEN"),
    )
    model.eval()
    return tokenizer, model

def clear_model(tokenizer=None, model=None):
    del tokenizer
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


## Appendix B-Aligned Llama 2 Chain-of-Thought Prompt

In [ ]:
PROMPT_POLICY_VERSION = "universal-final-policy-v2"

LLAMA2_COT_REQUESTS = [
    "You are an expert annotator for research-oriented, non-clinical emotion classification. Assist in analyzing emotions in text data. Do not make a clinical diagnosis, infer a medical condition, or provide treatment advice.",
    """I will first provide a piece of text. Independently assess its emotional content before comparing it with a Phase 1 AI-generated label. The only permitted labels are Depression, Neutral, and Happy.""",
    """Independently analyze the text using the classification policy below. State the dominant emotion and the textual evidence before considering the Phase 1 label.

Classification Policy:
- Depression: unresolved sadness, hopelessness, emotional distress, emotional exhaustion, withdrawal, self-devaluation, or a clearly negative overall trajectory is dominant.
- Neutral: the text is mainly factual, routine, balanced, informational, or emotionally mild, without a dominant positive or distress-related state.
- Happy: happiness, relief, gratitude, accomplishment, fulfillment, or a clearly positive resolution is dominant.

For every text, assess the dominant emotional meaning of the full text. Do not decide from isolated words, brief cues, or a simple average of positive and negative expressions. When the text contains multiple emotional cues or a clear temporal emotional shift, additionally consider the overall trajectory and final takeaway. Do not assume a trajectory when the text does not clearly support one. Do not select Neutral merely because multiple cues are present. A brief positive cue does not make the text Happy when unresolved distress remains dominant, and a brief negative cue does not make the text Depression when the text clearly resolves into sustained relief or positive resolution.

Use only Depression, Neutral, or Happy for your provisional label.""",
    """The Phase 1 classifier predicted: {phase1_label}

Compare that prediction with your independent assessment. Confirm it only when it is supported by the dominant emotional meaning of the full text. Otherwise, explain the correction using textual evidence only.""",
    """Provide the final Phase 2 decision. Use only one exact label: Depression, Neutral, or Happy. Do not use synonyms or additional labels such as Sad, Positive, Mixed, Anxiety, or Other. Do not provide a percentage breakdown.

End the response with exactly one line: Final label: [label]""",
]


def format_llama2_text_input(text):
    return f"Text:\n{text}"


In [ ]:
def run_llama2_cot_one(tokenizer, model, text, phase1_label):
    messages = [
        {"role": "system", "content": LLAMA2_COT_REQUESTS[0]},
        {"role": "user", "content": LLAMA2_COT_REQUESTS[1]},
    ]
    messages.append({"role": "assistant", "content": chat_generate(tokenizer, model, messages, MAX_NEW_TOKENS_COT, do_sample=True, temperature=0.6, top_p=0.9)})
    messages.append({"role": "user", "content": format_llama2_text_input(text)})
    messages.append({"role": "assistant", "content": chat_generate(tokenizer, model, messages, MAX_NEW_TOKENS_COT, do_sample=True, temperature=0.6, top_p=0.9)})

    independent_prompt = LLAMA2_COT_REQUESTS[2]
    messages.append({"role": "user", "content": independent_prompt})
    independent_answer = chat_generate(tokenizer, model, messages, MAX_NEW_TOKENS_COT, do_sample=True, temperature=0.6, top_p=0.9)
    messages.append({"role": "assistant", "content": independent_answer})

    comparison_prompt = LLAMA2_COT_REQUESTS[3].format(phase1_label=phase1_label)
    messages.append({"role": "user", "content": comparison_prompt})
    comparison_answer = chat_generate(tokenizer, model, messages, MAX_NEW_TOKENS_COT, do_sample=True, temperature=0.6, top_p=0.9)
    messages.append({"role": "assistant", "content": comparison_answer})

    messages.append({"role": "user", "content": LLAMA2_COT_REQUESTS[4]})
    final_answer = chat_generate(tokenizer, model, messages, MAX_NEW_TOKENS_COT, do_sample=True, temperature=0.6, top_p=0.9)
    return [independent_answer, comparison_answer, final_answer]


def parse_llama2_final_label(output):
    text = str(output)
    match = re.search(r"Final label\s*:\s*(Depression|Neutral|Happy)", text, flags=re.IGNORECASE)
    if match:
        return match.group(1).capitalize()

    fallback_patterns = [
        r"final (?:phase 2 )?(?:classification )?label[^.\n:]*[:\s]+(Depression|Neutral|Happy)",
        r"correct classification[^.\n:]*[:\s]+(Depression|Neutral|Happy)",
        r"dominant emotion[^.\n:]*[:\s]+(Depression|Neutral|Happy)",
        r"classif(?:y|ied|ication)[^.\n]*\b(Depression|Neutral|Happy)\b",
    ]
    for pattern in fallback_patterns:
        fallback = re.search(pattern, text, flags=re.IGNORECASE)
        if fallback:
            return fallback.group(1).capitalize()
    return np.nan


In [ ]:
def _normalize_id(value):
    if pd.isna(value):
        return None
    return str(value)


def load_existing_results(path):
    path = Path(path)
    if RESUME_FROM_EXISTING and path.exists() and path.stat().st_size > 0:
        existing = pd.read_csv(path)
        if "example_id" in existing.columns:
            existing = existing.drop_duplicates(subset=["example_id"], keep="last")
        return existing
    return pd.DataFrame()


def completed_example_ids(path):
    existing = load_existing_results(path)
    if existing.empty or "example_id" not in existing.columns:
        return set()
    return set(existing["example_id"].map(_normalize_id).dropna())


def append_result_row(path, row_dict):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame([row_dict]).to_csv(
        path,
        mode="a",
        header=not path.exists(),
        index=False,
    )


In [ ]:
llama2_results = load_existing_results(LLAMA2_OUTPUT_PATH)

if RUN_LLAMA2_COT:
    completed_ids = completed_example_ids(LLAMA2_OUTPUT_PATH)
    pending_df = work_df[~work_df["example_id"].map(_normalize_id).isin(completed_ids)].copy()
    print(f"Llama 2 CoT resume: {len(completed_ids)} completed, {len(pending_df)} pending.")

    if pending_df.empty:
        print(f"No pending Llama 2 rows. Loaded existing results from: {LLAMA2_OUTPUT_PATH}")
    else:
        tokenizer, model = load_chat_model(LLAMA2_MODEL_NAME, load_in_4bit=True)
        for _, row in tqdm(pending_df.iterrows(), total=len(pending_df), desc="Llama 2 CoT"):
            answers = run_llama2_cot_one(
                tokenizer,
                model,
                row[TEXT_COL],
                row["phase1_label_for_prompt"],
            )
            result_row = {
                "example_id": row.get("example_id", None),
                "text": row[TEXT_COL],
                "target_label": row[TRUE_LABEL_COL],
                "phase1_label_for_prompt": row["phase1_label_for_prompt"],
                "prompt_policy_version": PROMPT_POLICY_VERSION,
                "LLaMA2_1": answers[0] if len(answers) > 0 else np.nan,
                "LLaMA2_2": answers[1] if len(answers) > 1 else np.nan,
                "LLaMA2_3": answers[2] if len(answers) > 2 else np.nan,
            }
            result_row["LLaMA2_final_label"] = parse_llama2_final_label(result_row["LLaMA2_3"])
            append_result_row(LLAMA2_OUTPUT_PATH, result_row)
        clear_model(tokenizer, model)

    llama2_results = load_existing_results(LLAMA2_OUTPUT_PATH)
    print(f"Saved/resumed: {LLAMA2_OUTPUT_PATH}")
    print(f"Llama 2 rows available: {len(llama2_results)}")
    display(llama2_results.head())


## Appendix C-Aligned Llama 3 SELF-DISCOVER Prompt

The SELF-DISCOVER select/adapt/implement prompt templates are embedded here from `prompts.py`, so this notebook does not need a separate `prompts.py` upload in Colab.


In [ ]:
reasoning_modules = """
1 How could I devise an experiment to help solve that problem?
2 Make a list of ideas for solving this problem, and apply them one by one to the problem to see if any progress can be made.
3 How could I measure progress on this problem?
4 How can I simplify the problem so that it is easier to solve?
5 What are the key assumptions underlying this problem?
6 What are the potential risks and drawbacks of each solution?
7 What are the alternative perspectives or viewpoints on this problem?
8 What are the long-term implications of this problem and its solutions?
9 How can I break down this problem into smaller, more manageable parts?
10 Critical Thinking: This style involves analyzing the problem from different perspectives, questioning assumptions, and evaluating
the evidence or information available. It focuses on logical reasoning, evidence-based decision-making, and identifying
potential biases or flaws in thinking.
11 Try creative thinking, generate innovative and out-of-the-box ideas to solve the problem. Explore unconventional solutions,
thinking beyond traditional boundaries, and encouraging imagination and originality.
12 Seek input and collaboration from others to solve the problem. Emphasize teamwork, open communication, and leveraging the
diverse perspectives and expertise of a group to come up with effective solutions.
13 Use systems thinking: Consider the problem as part of a larger system and understanding the interconnectedness of various elements.
Focuses on identifying the underlying causes, feedback loops, and interdependencies that influence the problem, and developing holistic
solutions that address the system as a whole.
14 Use Risk Analysis: Evaluate potential risks, uncertainties, and tradeoffs associated with different solutions or approaches to a
problem. Emphasize assessing the potential consequences and likelihood of success or failure, and making informed decisions based
on a balanced analysis of risks and benefits.
15 Use Reflective Thinking: Step back from the problem, take the time for introspection and self-reflection. Examine personal biases,
assumptions, and mental models that may influence problem-solving, and being open to learning from past experiences to improve
future approaches.
16 What is the core issue or problem that needs to be addressed?
17 What are the underlying causes or factors contributing to the problem?
18 Are there any potential solutions or strategies that have been tried before? If yes, what were the outcomes and lessons learned?
19 What are the potential obstacles or challenges that might arise in solving this problem?
20 Are there any relevant data or information that can provide insights into the problem? If yes, what data sources are available,
and how can they be analyzed?
21 Are there any stakeholders or individuals who are directly affected by the problem? What are their perspectives and needs?
22 What resources (financial, human, technological, etc.) are needed to tackle the problem effectively?
23 How can progress or success in solving the problem be measured or evaluated?
24 What indicators or metrics can be used?
25 Is the problem a technical or practical one that requires a specific expertise or skill set? Or is it more of a conceptual or
theoretical problem?
26 Does the problem involve a physical constraint, such as limited resources, infrastructure, or space?
27 Is the problem related to human behavior, such as a social, cultural, or psychological issue?
28 Does the problem involve decision-making or planning, where choices need to be made under uncertainty or with competing
objectives?
29 Is the problem an analytical one that requires data analysis, modeling, or optimization techniques?
30 Is the problem a design challenge that requires creative solutions and innovation?
31 Does the problem require addressing systemic or structural issues rather than just individual instances?
32 Is the problem time-sensitive or urgent, requiring immediate attention and action?
33 What kinds of solution typically are produced for this kind of problem specification?
34 Given the problem specification and the current best solution, have a guess about other possible solutions.
35 Let's imagine the current best solution is totally wrong, what other ways are there to think about the problem specification?
36 What is the best way to modify this current best solution, given what you know about these kinds of problem specification?
37 Ignoring the current best solution, create an entirely new solution to the problem.
38 Let's think step by step.
39 Let's make a step by step plan and implement it with good notion and explanation"""


select_prompt = """
In order to solve the given task:
<Task>
{Task}
</Task>
Select several modules that are crucial for solving the tasks above
from all the reasoning module description given below:
{resonining_modules}
"""

adapt_prompt = """
Rephrase and specify each reasoning module so that it better helps solving the task:
<Task>
{Task}
</Task>
SELECTED module descriptions:
{selected_modules}
Adapt each reasoning module description to better solve the task:
"""

implement_prompt = """
Operationalize the reasoning modules into a step-by-step reasoning plan in JSON format
Example task:
<Task>
{Task}
</Task>
ADAPTED module descriptions:
{adapted_modules}

Implement a reasoning structure to generalise similar task to follow step-by-step and arrive at correct answers
"""

In [ ]:
SELF_DISCOVER_TASK_TEMPLATE = """
You are an expert annotator for research-oriented, non-clinical emotion classification.
Your task is to analyze the emotional content of text data using structured reasoning.
Do not make a clinical diagnosis, infer a medical condition, or provide treatment advice.

<context>
data: {data}
phase_1_label: {label}
</context>

<questions>
1. Before considering the Phase 1 label, independently analyze the given text. Identify the dominant emotional meaning of the full text and cite the relevant textual evidence.
2. Determine whether the text contains multiple emotional cues or a clear temporal emotional shift. Apply trajectory reasoning only when the text supports such a shift; otherwise do not invent one.
3. Compare the Phase 1 label with your independent assessment. Confirm it only if it is supported by the text; otherwise correct it using textual evidence only.
4. Select exactly one final label from Depression, Neutral, and Happy.

Classification Policy:
- Depression: unresolved sadness, hopelessness, emotional distress, emotional exhaustion, withdrawal, self-devaluation, or a clearly negative overall trajectory is dominant.
- Neutral: the text is mainly factual, routine, balanced, informational, or emotionally mild, without a dominant positive or distress-related state.
- Happy: happiness, relief, gratitude, accomplishment, fulfillment, or a clearly positive resolution is dominant.

Mixed and Shifting Emotion Rule:
- Assess the dominant emotional meaning of the full text; do not classify from isolated phrases, brief cues, or a simple average of emotional words.
- When multiple emotional cues or a clear temporal shift is present, use the overall trajectory and final takeaway to resolve the label.
- Do not select Neutral merely because multiple cues are present.
- Do not select Happy from a brief positive cue when unresolved distress remains dominant.
- Do not select Depression from a brief negative cue when the text clearly resolves into sustained relief or positive resolution.

Output Constraint:
Use only Depression, Neutral, or Happy. Do not output synonyms or other labels such as Sad, Positive, Mixed, Anxiety, or Other. Base the justification on textual evidence rather than clinical assumptions. End with exactly one final label in the format Final label: [label].
</questions>
"""

FIXED_SELF_DISCOVER_STRUCTURE = """
Use a structured, text-grounded reasoning plan:
1. Independently identify the dominant emotional cues in the full text.
2. Determine whether a blended or shifting emotional trajectory is explicitly supported.
3. Apply trajectory reasoning only when such a shift is present; otherwise rely on dominant emotional meaning.
4. Compare the independent assessment with the AI-generated Phase 1 label.
5. Justify the decision using textual evidence only and choose only Depression, Neutral, or Happy.
6. End with exactly one label in the format Final label: [label].
"""

def build_self_discover_task(text, phase1_label):
    return SELF_DISCOVER_TASK_TEMPLATE.replace("{data}", str(text)).replace("{label}", str(phase1_label))

def parse_llama3_final_label(output):
    text = str(output)
    match = re.search(r"Final label\s*:\s*(Depression|Neutral|Happy)", text, flags=re.IGNORECASE)
    if match:
        value = match.group(1).lower()
        return next(label for label in LABELS if label.lower() == value)
    for label in LABELS:
        if re.search(rf"\b{label}\b", text, flags=re.IGNORECASE):
            return label
    return np.nan


In [ ]:
# Llama 3 can be run after a partial Colab execution. If the shared chat helper
# cell above was skipped, define the same helper functions here as a fallback.
if "chat_generate" not in globals():
    def format_messages_without_chat_template(messages):
        """Fallback formatter for chat models whose tokenizer has no chat_template."""
        system_parts = [m["content"] for m in messages if m.get("role") == "system"]
        dialogue = [m for m in messages if m.get("role") != "system"]
        system_text = "\n".join(system_parts).strip()

        prompt = ""
        pending_user = None
        first_turn = True
        for message in dialogue:
            role = message.get("role")
            content = str(message.get("content", "")).strip()
            if role == "user":
                pending_user = content
            elif role == "assistant" and pending_user is not None:
                if first_turn and system_text:
                    prompt += f"<s>[INST] <<SYS>>\n{system_text}\n<</SYS>>\n\n{pending_user} [/INST] {content} </s>"
                else:
                    prompt += f"<s>[INST] {pending_user} [/INST] {content} </s>"
                pending_user = None
                first_turn = False

        if pending_user is not None:
            if first_turn and system_text:
                prompt += f"<s>[INST] <<SYS>>\n{system_text}\n<</SYS>>\n\n{pending_user} [/INST]"
            else:
                prompt += f"<s>[INST] {pending_user} [/INST]"
        return prompt

    def build_chat_inputs(tokenizer, model, messages):
        if getattr(tokenizer, "chat_template", None):
            prompt = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )
        else:
            prompt = format_messages_without_chat_template(messages)

        encoded = tokenizer(prompt, return_tensors="pt")
        return {key: value.to(model.device) for key, value in encoded.items()}

    def chat_generate(tokenizer, model, messages, max_new_tokens=256, do_sample=False, temperature=None, top_p=None):
        model_inputs = build_chat_inputs(tokenizer, model, messages)
        input_ids = model_inputs["input_ids"]

        terminators = [tokenizer.eos_token_id]
        eot_id = tokenizer.convert_tokens_to_ids("<|eot_id|>")
        if isinstance(eot_id, int) and eot_id >= 0:
            terminators.append(eot_id)

        kwargs = dict(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            eos_token_id=terminators,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=do_sample,
        )
        if temperature is not None:
            kwargs["temperature"] = temperature
        if top_p is not None:
            kwargs["top_p"] = top_p

        with torch.no_grad():
            outputs = model.generate(**kwargs)
        response = outputs[0][input_ids.shape[-1]:]
        return tokenizer.decode(response, skip_special_tokens=True).strip()

def run_self_discover_structure(tokenizer, model, task):
    select = select_prompt.replace("{Task}", task).replace("{resonining_modules}", reasoning_modules)
    selected_modules = chat_generate(tokenizer, model, [{"role": "user", "content": select}], MAX_NEW_TOKENS_SELF_DISCOVER, do_sample=True, top_p=1.0)

    adapt = adapt_prompt.replace("{Task}", task).replace("{selected_modules}", selected_modules)
    adapted_modules = chat_generate(tokenizer, model, [{"role": "user", "content": adapt}], MAX_NEW_TOKENS_SELF_DISCOVER, do_sample=True, top_p=1.0)

    implement = implement_prompt.replace("{Task}", task).replace("{adapted_modules}", adapted_modules)
    reasoning_structure = chat_generate(tokenizer, model, [{"role": "user", "content": implement}], MAX_NEW_TOKENS_SELF_DISCOVER, do_sample=True, top_p=1.0)
    return selected_modules, adapted_modules, reasoning_structure

def run_llama3_self_discover_one(tokenizer, model, text, phase1_label):
    task = build_self_discover_task(text, phase1_label)
    if SELF_DISCOVER_STRUCTURE_MODE == "per_sample":
        selected, adapted, structure = run_self_discover_structure(tokenizer, model, task)
    elif SELF_DISCOVER_STRUCTURE_MODE == "fixed":
        selected = "Fixed paper-safe reasoning modules: textual evidence review, label comparison, mixed/shifting emotion handling, final trajectory assessment, and final constrained label selection."
        adapted = "Adapted to identify dominant emotional cues, prioritize final emotional trajectory for blended texts, compare with the AI-generated label, and avoid clinical inference."
        structure = FIXED_SELF_DISCOVER_STRUCTURE
    else:
        raise ValueError("SELF_DISCOVER_STRUCTURE_MODE must be 'per_sample' or 'fixed'.")

    final_prompt = (
        f"Using the following reasoning structure:\n{structure}\n\n"
        f"Solve this task, providing your answer:\n{task}\n\n"
        "Note1: Write the question number before your answer.\n"
        "Note2: Do not write anything besides your answer.\n"
        "Note3: End with exactly one final label in the format Final label: Depression, Final label: Neutral, or Final label: Happy."
    )
    messages = [
        {"role": "system", "content": "You are an expert annotator for research-oriented, non-clinical emotion classification. Do not provide clinical diagnosis, medical inference, treatment advice, or professional mental health advice."},
        {"role": "user", "content": final_prompt},
    ]
    answer = chat_generate(tokenizer, model, messages, MAX_NEW_TOKENS_SELF_DISCOVER, do_sample=False)
    return selected, adapted, structure, answer


In [ ]:
llama3_results = load_existing_results(LLAMA3_OUTPUT_PATH)

if RUN_LLAMA3_SELF_DISCOVER:
    completed_ids = completed_example_ids(LLAMA3_OUTPUT_PATH)
    pending_df = work_df[~work_df["example_id"].map(_normalize_id).isin(completed_ids)].copy()
    print(f"Llama 3 SELF-DISCOVER resume: {len(completed_ids)} completed, {len(pending_df)} pending.")

    if pending_df.empty:
        print(f"No pending Llama 3 rows. Loaded existing results from: {LLAMA3_OUTPUT_PATH}")
    else:
        tokenizer, model = load_chat_model(LLAMA3_MODEL_NAME, load_in_4bit=True)
        for _, row in tqdm(pending_df.iterrows(), total=len(pending_df), desc="Llama 3 SELF-DISCOVER"):
            selected, adapted, structure, answer = run_llama3_self_discover_one(
                tokenizer,
                model,
                row[TEXT_COL],
                row["phase1_label_for_prompt"],
            )
            result_row = {
                "example_id": row.get("example_id", None),
                "text": row[TEXT_COL],
                "target_label": row[TRUE_LABEL_COL],
                "phase1_label_for_prompt": row["phase1_label_for_prompt"],
                "prompt_policy_version": PROMPT_POLICY_VERSION,
                "LLaMA3_SELECT": selected,
                "LLaMA3_ADAPT": adapted,
                "LLaMA3_IMPLEMENT": structure,
                "LLaMA3_Answer": answer,
            }
            result_row["LLaMA3_final_label"] = parse_llama3_final_label(result_row["LLaMA3_Answer"])
            append_result_row(LLAMA3_OUTPUT_PATH, result_row)
        clear_model(tokenizer, model)

    llama3_results = load_existing_results(LLAMA3_OUTPUT_PATH)
    print(f"Saved/resumed: {LLAMA3_OUTPUT_PATH}")
    print(f"Llama 3 rows available: {len(llama3_results)}")
    display(llama3_results.head())



## Routed Subset Phase 2 Evaluation

This section evaluates Phase 2 final labels only on the routed Reddit held-out test rows. The full Reddit held-out end-to-end evaluation is computed later by combining accepted Phase 1 labels with routed Phase 2 labels.



In [ ]:
def _safe_classification_report(eval_df, pred_col, model_name):
    if classification_report is None or eval_df.empty:
        return pd.DataFrame()
    report = classification_report(
        eval_df["target_label"],
        eval_df[pred_col],
        labels=LABELS,
        output_dict=True,
        zero_division=0,
    )
    rows = []
    for label, values in report.items():
        if isinstance(values, dict):
            row = {"model": model_name, "label": label, **values}
        else:
            row = {"model": model_name, "label": label, "score": values}
        rows.append(row)
    return pd.DataFrame(rows)


def _save_phase2_confusion_matrix(eval_df, pred_col, model_name, filename_prefix):
    if confusion_matrix is None or eval_df.empty:
        return None, pd.DataFrame()
    cm = confusion_matrix(eval_df["target_label"], eval_df[pred_col], labels=LABELS)
    cm_df = pd.DataFrame(cm, index=LABELS, columns=LABELS)
    cm_csv_path = OUTPUT_DIR / f"{filename_prefix}_confusion_matrix.csv"
    cm_df.to_csv(cm_csv_path)

    plt.figure(figsize=(5.2, 4.2))
    sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues")
    plt.xlabel("Predicted")
    plt.ylabel("Target")
    plt.title(f"{model_name} Phase 2 Confusion Matrix")
    plt.tight_layout()
    png_path = OUTPUT_DIR / f"{filename_prefix}_confusion_matrix.png"
    plt.savefig(png_path, dpi=220)
    plt.show()
    return png_path, cm_df


def evaluate_final_labels(result_df, pred_col, name, filename_prefix):
    if result_df is None or pred_col not in result_df.columns:
        print(f"{name}: no results to evaluate.")
        return None, pd.DataFrame(), pd.DataFrame()
    eval_df = result_df.dropna(subset=[pred_col]).copy()
    if eval_df.empty:
        print(f"{name}: no parseable final labels.")
        return None, pd.DataFrame(), pd.DataFrame()

    acc = (eval_df[pred_col] == eval_df["target_label"]).mean()
    label_counts = eval_df[pred_col].value_counts().reindex(LABELS, fill_value=0).rename_axis("predicted_label").reset_index(name="count")
    label_counts.insert(0, "model", name)
    label_counts.to_csv(OUTPUT_DIR / f"{filename_prefix}_predicted_label_counts.csv", index=False)

    print(f"\n{name}")
    print(f"Rows evaluated: {len(eval_df)} / {len(result_df)}")
    print(f"Accuracy vs target_label: {acc:.4f}")
    cm_path, cm_df = _save_phase2_confusion_matrix(eval_df, pred_col, name, filename_prefix)
    display(cm_df)
    display(label_counts)

    report_df = _safe_classification_report(eval_df, pred_col, name)
    if not report_df.empty:
        report_path = OUTPUT_DIR / f"{filename_prefix}_classification_report.csv"
        report_df.to_csv(report_path, index=False)
        display(report_df)

    parse_failures = result_df[result_df[pred_col].isna()].copy()
    if not parse_failures.empty:
        parse_failures.to_csv(OUTPUT_DIR / f"{filename_prefix}_parse_failures.csv", index=False)

    return {
        "model": name,
        "rows_total": len(result_df),
        "rows_evaluated": len(eval_df),
        "parse_failures": int(result_df[pred_col].isna().sum()),
        "accuracy": float(acc),
        "confusion_matrix_png": str(cm_path) if cm_path else None,
    }, report_df, label_counts

summary = []
all_reports = []
all_label_counts = []
item, report, counts = evaluate_final_labels(llama2_results, "LLaMA2_final_label", "Llama 2 CoT", "llama2_cot")
if item:
    summary.append(item)
if not report.empty:
    all_reports.append(report)
if not counts.empty:
    all_label_counts.append(counts)

item, report, counts = evaluate_final_labels(llama3_results, "LLaMA3_final_label", "Llama 3 SELF-DISCOVER", "llama3_self_discover")
if item:
    summary.append(item)
if not report.empty:
    all_reports.append(report)
if not counts.empty:
    all_label_counts.append(counts)

summary_df = pd.DataFrame(summary)
if not summary_df.empty:
    summary_path = OUTPUT_DIR / "reddit_test_phase2_llm_reasoning_summary.csv"
    summary_df.to_csv(summary_path, index=False)
    print(f"Saved: {summary_path}")
    display(summary_df)

if all_reports:
    phase2_reports_df = pd.concat(all_reports, ignore_index=True)
    phase2_reports_df.to_csv(OUTPUT_DIR / "reddit_test_phase2_classification_reports.csv", index=False)
if all_label_counts:
    phase2_label_counts_df = pd.concat(all_label_counts, ignore_index=True)
    phase2_label_counts_df.to_csv(OUTPUT_DIR / "reddit_test_phase2_predicted_label_counts.csv", index=False)




## Combined Output Table

In [ ]:
combined = work_df[["example_id", TEXT_COL, TRUE_LABEL_COL, "phase1_label_for_prompt"]].copy()
if llama2_results is not None:
    combined = combined.merge(
        llama2_results[["example_id", "LLaMA2_1", "LLaMA2_2", "LLaMA2_3", "LLaMA2_final_label"]],
        on="example_id",
        how="left",
    )
if llama3_results is not None:
    combined = combined.merge(
        llama3_results[["example_id", "LLaMA3_SELECT", "LLaMA3_ADAPT", "LLaMA3_IMPLEMENT", "LLaMA3_Answer", "LLaMA3_final_label"]],
        on="example_id",
        how="left",
    )
combined_path = OUTPUT_DIR / "reddit_test_phase2_reasoning_universal_prompt_v2_combined_outputs.csv"
combined.to_csv(combined_path, index=False)
print(f"Saved: {combined_path}")
display(combined.head())



## Reddit Held-Out Test End-to-End Orchestration

This section merges the routed Llama outputs back into the full Reddit held-out test set. Accepted rows keep their Phase 1 DistilBERT label; routed rows use the parsed Phase 2 label when available.


In [ ]:
def normalize_label(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    for label in LABELS:
        if text.lower() == label.lower():
            return label
    for label in LABELS:
        if label.lower() in text.lower():
            return label
    return np.nan


def metrics_for(dataframe, pred_col, name):
    valid = dataframe.dropna(subset=[TRUE_LABEL_COL, pred_col]).copy()
    if valid.empty:
        return {"model": name, "rows": 0, "accuracy": np.nan, "macro_precision": np.nan, "macro_recall": np.nan, "macro_f1": np.nan}
    p, r, f1, _ = precision_recall_fscore_support(valid[TRUE_LABEL_COL], valid[pred_col], labels=LABELS, average="macro", zero_division=0)
    return {
        "model": name,
        "rows": len(valid),
        "accuracy": accuracy_score(valid[TRUE_LABEL_COL], valid[pred_col]),
        "macro_precision": p,
        "macro_recall": r,
        "macro_f1": f1,
    }


def classification_report_frame(dataframe, pred_col, name):
    valid = dataframe.dropna(subset=[TRUE_LABEL_COL, pred_col]).copy()
    if valid.empty or classification_report is None:
        return pd.DataFrame()
    report = classification_report(valid[TRUE_LABEL_COL], valid[pred_col], labels=LABELS, output_dict=True, zero_division=0)
    rows = []
    for label, values in report.items():
        if isinstance(values, dict):
            rows.append({"model": name, "label": label, **values})
        else:
            rows.append({"model": name, "label": label, "score": values})
    return pd.DataFrame(rows)


def confusion_matrix_frame(dataframe, pred_col, name):
    valid = dataframe.dropna(subset=[TRUE_LABEL_COL, pred_col]).copy()
    cm = confusion_matrix(valid[TRUE_LABEL_COL], valid[pred_col], labels=LABELS)
    cm_df = pd.DataFrame(cm, index=LABELS, columns=LABELS)
    cm_df.index.name = "target_label"
    cm_df.columns.name = "predicted_label"
    long_df = cm_df.reset_index().melt(id_vars="target_label", var_name="predicted_label", value_name="count")
    long_df.insert(0, "model", name)
    return cm_df, long_df


def save_cm(dataframe, pred_col, title, filename):
    cm_df, _ = confusion_matrix_frame(dataframe, pred_col, title)
    csv_path = OUTPUT_DIR / filename.replace(".png", ".csv")
    cm_df.to_csv(csv_path)
    plt.figure(figsize=(5.2, 4.2))
    sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues")
    plt.xlabel("Predicted")
    plt.ylabel("Target")
    plt.title(title)
    plt.tight_layout()
    png_path = OUTPUT_DIR / filename
    plt.savefig(png_path, dpi=220)
    plt.show()
    return png_path


base = df.copy()
base[PHASE1_LABEL_COL] = base[PHASE1_LABEL_COL].map(normalize_label)
base[TRUE_LABEL_COL] = base[TRUE_LABEL_COL].map(normalize_label)

if llama2_results is not None and not llama2_results.empty:
    base = base.merge(
        llama2_results[["example_id", "LLaMA2_final_label"]].drop_duplicates("example_id", keep="last"),
        on="example_id",
        how="left",
    )
    base["LLaMA2_final_label"] = base["LLaMA2_final_label"].map(normalize_label)
else:
    base["LLaMA2_final_label"] = np.nan

if llama3_results is not None and not llama3_results.empty:
    base = base.merge(
        llama3_results[["example_id", "LLaMA3_final_label"]].drop_duplicates("example_id", keep="last"),
        on="example_id",
        how="left",
    )
    base["LLaMA3_final_label"] = base["LLaMA3_final_label"].map(normalize_label)
else:
    base["LLaMA3_final_label"] = np.nan

base["final_label_llama2"] = np.where(base[PHASE1_ROUTED_COL] & base["LLaMA2_final_label"].notna(), base["LLaMA2_final_label"], base[PHASE1_LABEL_COL])
base["final_source_llama2"] = np.where(base[PHASE1_ROUTED_COL] & base["LLaMA2_final_label"].notna(), "llama2_cot", "phase1")
base["final_label_llama3"] = np.where(base[PHASE1_ROUTED_COL] & base["LLaMA3_final_label"].notna(), base["LLaMA3_final_label"], base[PHASE1_LABEL_COL])
base["final_source_llama3"] = np.where(base[PHASE1_ROUTED_COL] & base["LLaMA3_final_label"].notna(), "llama3_self_discover", "phase1")

base["is_correct_phase1"] = base[PHASE1_LABEL_COL] == base[TRUE_LABEL_COL]
base["is_correct_llama2_e2e"] = base["final_label_llama2"] == base[TRUE_LABEL_COL]
base["is_correct_llama3_e2e"] = base["final_label_llama3"] == base[TRUE_LABEL_COL]

end_to_end_path = OUTPUT_DIR / "reddit_test_end_to_end_results.csv"
base.to_csv(end_to_end_path, index=False)
print("Saved:", end_to_end_path)

metric_specs = [(PHASE1_LABEL_COL, "Phase 1 DistilBERT only")]
if base["LLaMA2_final_label"].notna().any():
    metric_specs.append(("LLaMA2_final_label", "Llama 2 routed only"))
    metric_specs.append(("final_label_llama2", "Reddit end-to-end with Llama 2"))
if base["LLaMA3_final_label"].notna().any():
    metric_specs.append(("LLaMA3_final_label", "Llama 3 routed only"))
    metric_specs.append(("final_label_llama3", "Reddit end-to-end with Llama 3"))

metrics_df = pd.DataFrame([metrics_for(base[base[PHASE1_ROUTED_COL]] if "routed only" in name else base, col, name) for col, name in metric_specs])
metrics_df.to_csv(OUTPUT_DIR / "reddit_test_end_to_end_metrics_summary.csv", index=False)
display(metrics_df)

report_frames = []
cm_long_frames = []
figure_paths = []
for col, name in metric_specs:
    eval_base = base[base[PHASE1_ROUTED_COL]] if "routed only" in name else base
    report = classification_report_frame(eval_base, col, name)
    if not report.empty:
        report_frames.append(report)
    _, cm_long = confusion_matrix_frame(eval_base, col, name)
    cm_long_frames.append(cm_long)

classification_reports_df = pd.concat(report_frames, ignore_index=True) if report_frames else pd.DataFrame()
classification_reports_df.to_csv(OUTPUT_DIR / "reddit_test_end_to_end_classification_reports.csv", index=False)
confusion_matrices_long_df = pd.concat(cm_long_frames, ignore_index=True) if cm_long_frames else pd.DataFrame()
confusion_matrices_long_df.to_csv(OUTPUT_DIR / "reddit_test_end_to_end_confusion_matrices_long.csv", index=False)

correction_rows = []
for model_name, final_col, phase2_col in [("llama2", "final_label_llama2", "LLaMA2_final_label"), ("llama3", "final_label_llama3", "LLaMA3_final_label")]:
    if phase2_col not in base or not base[phase2_col].notna().any():
        continue
    routed = base[base[PHASE1_ROUTED_COL]].copy()
    corrected = (~routed["is_correct_phase1"]) & (routed[phase2_col] == routed[TRUE_LABEL_COL])
    introduced = (routed["is_correct_phase1"]) & (routed[phase2_col].notna()) & (routed[phase2_col] != routed[TRUE_LABEL_COL])
    remaining = (~routed["is_correct_phase1"]) & (routed[phase2_col] != routed[TRUE_LABEL_COL])
    correction_rows.append({
        "phase2_model": model_name,
        "routed_rows": len(routed),
        "phase1_errors_in_routed": int((~routed["is_correct_phase1"]).sum()),
        "corrected_errors": int(corrected.sum()),
        "introduced_errors": int(introduced.sum()),
        "remaining_routed_errors": int(remaining.sum()),
        "net_error_reduction": int(corrected.sum()) - int(introduced.sum()),
    })
correction_df = pd.DataFrame(correction_rows)
correction_df.to_csv(OUTPUT_DIR / "reddit_test_phase2_correction_analysis.csv", index=False)
display(correction_df)

routing_summary = pd.DataFrame([{
    "total_rows": len(base),
    "accepted_by_phase1": int(base[PHASE1_ACCEPTED_COL].sum()),
    "routed_to_phase2": int(base[PHASE1_ROUTED_COL].sum()),
    "coverage": float(base[PHASE1_ACCEPTED_COL].mean()),
    "routing_rate": float(base[PHASE1_ROUTED_COL].mean()),
    "phase1_accuracy_all": float((base[PHASE1_LABEL_COL] == base[TRUE_LABEL_COL]).mean()),
    "phase1_accuracy_accepted": float((base.loc[base[PHASE1_ACCEPTED_COL], PHASE1_LABEL_COL] == base.loc[base[PHASE1_ACCEPTED_COL], TRUE_LABEL_COL]).mean()) if base[PHASE1_ACCEPTED_COL].any() else np.nan,
    "phase1_accuracy_routed": float((base.loc[base[PHASE1_ROUTED_COL], PHASE1_LABEL_COL] == base.loc[base[PHASE1_ROUTED_COL], TRUE_LABEL_COL]).mean()) if base[PHASE1_ROUTED_COL].any() else np.nan,
    "routing_threshold": float(base["routing_threshold"].dropna().iloc[0]) if "routing_threshold" in base and base["routing_threshold"].notna().any() else np.nan,
    "temperature": float(base["temperature"].dropna().iloc[0]) if "temperature" in base and base["temperature"].notna().any() else np.nan,
}])
routing_summary.to_csv(OUTPUT_DIR / "reddit_test_routing_coverage_table.csv", index=False)
display(routing_summary)

label_distribution_rows = []
for col, name in metric_specs:
    eval_base = base[base[PHASE1_ROUTED_COL]] if "routed only" in name else base
    counts = eval_base[col].value_counts(dropna=False).rename_axis("label").reset_index(name="count")
    counts.insert(0, "model", name)
    label_distribution_rows.append(counts)
label_distribution_df = pd.concat(label_distribution_rows, ignore_index=True)
label_distribution_df.to_csv(OUTPUT_DIR / "reddit_test_prediction_label_distributions.csv", index=False)

figure_paths.append(save_cm(base, PHASE1_LABEL_COL, "Reddit Test Phase 1 DistilBERT", "reddit_test_confusion_matrix_phase1.png"))
if base["LLaMA2_final_label"].notna().any():
    figure_paths.append(save_cm(base, "final_label_llama2", "Reddit Test End-to-End with Llama 2", "reddit_test_confusion_matrix_llama2_e2e.png"))
if base["LLaMA3_final_label"].notna().any():
    figure_paths.append(save_cm(base, "final_label_llama3", "Reddit Test End-to-End with Llama 3", "reddit_test_confusion_matrix_llama3_e2e.png"))

with pd.ExcelWriter(OUTPUT_DIR / "reddit_test_paper_ready_tables.xlsx") as writer:
    metrics_df.to_excel(writer, index=False, sheet_name="metrics_summary")
    routing_summary.to_excel(writer, index=False, sheet_name="routing_coverage")
    correction_df.to_excel(writer, index=False, sheet_name="correction_analysis")
    classification_reports_df.to_excel(writer, index=False, sheet_name="classification_reports")
    confusion_matrices_long_df.to_excel(writer, index=False, sheet_name="confusion_matrices_long")
    label_distribution_df.to_excel(writer, index=False, sheet_name="label_distributions")

print("Saved paper-ready workbook:", OUTPUT_DIR / "reddit_test_paper_ready_tables.xlsx")



## Reddit Test Final Paper Output Review

This section displays the main Reddit held-out end-to-end outputs in one place.


In [ ]:
print("=" * 80)
print("REDDIT HELD-OUT TEST END-TO-END REVIEW")
print("Output directory:", OUTPUT_DIR)
print("Rows:", len(base))
print("=" * 80)

print("\n[1] Metrics summary")
display(metrics_df)

print("\n[2] Routing / coverage summary")
display(routing_summary)

print("\n[3] Phase 2 correction analysis")
display(correction_df)

print("\n[4] Prediction label distributions")
display(label_distribution_df)

print("\n[5] Classification reports")
display(classification_reports_df)

print("\n[6] Confusion matrix figures")
for fig_path in figure_paths:
    if Path(fig_path).exists():
        print(fig_path)
        display(Image(filename=str(fig_path)))

print("\n[7] Example rows: Phase 1 routed errors")
routed_phase1_errors = base[base[PHASE1_ROUTED_COL] & (~base["is_correct_phase1"])].copy()
display(routed_phase1_errors[["example_id", TRUE_LABEL_COL, PHASE1_LABEL_COL, "phase1_confidence", TEXT_COL]].head(20))
routed_phase1_errors.to_csv(OUTPUT_DIR / "reddit_test_routed_phase1_error_examples.csv", index=False)

if base["LLaMA2_final_label"].notna().any():
    print("\n[8] Example rows: Llama 2 end-to-end errors")
    llama2_errors = base[~base["is_correct_llama2_e2e"]].copy()
    display(llama2_errors[["example_id", TRUE_LABEL_COL, PHASE1_LABEL_COL, "LLaMA2_final_label", "final_label_llama2", "final_source_llama2", TEXT_COL]].head(20))
    llama2_errors.to_csv(OUTPUT_DIR / "reddit_test_llama2_e2e_error_examples.csv", index=False)

if base["LLaMA3_final_label"].notna().any():
    print("\n[9] Example rows: Llama 3 end-to-end errors")
    llama3_errors = base[~base["is_correct_llama3_e2e"]].copy()
    display(llama3_errors[["example_id", TRUE_LABEL_COL, PHASE1_LABEL_COL, "LLaMA3_final_label", "final_label_llama3", "final_source_llama3", TEXT_COL]].head(20))
    llama3_errors.to_csv(OUTPUT_DIR / "reddit_test_llama3_e2e_error_examples.csv", index=False)

print("\n[10] Output files")
for p in sorted(OUTPUT_DIR.rglob("*")):
    if p.is_file():
        print(f"- {p.name} | {p.stat().st_size:,} bytes")



## Final export and local download

This cell packages all available result CSV files and starts a browser download.


In [ ]:
# Final export / download cell
# Run this after Llama 2 and/or Llama 3 cells finish.
# If OUTPUT_DIR is Google Drive, files are already persistent. This cell also creates one zip for easy local download.

from pathlib import Path
import zipfile

existing_output_files = []
for pattern in ["*.csv", "*.json", "*.png", "*.xlsx"]:
    existing_output_files.extend(sorted(OUTPUT_DIR.rglob(pattern)))
existing_output_files = sorted(set(Path(p) for p in existing_output_files))

print("Existing output files:")
for p in existing_output_files:
    print(f"- {p} | {p.stat().st_size:,} bytes")

if not existing_output_files:
    print("No output files found yet. Run the Llama 2/Llama 3 reasoning and Reddit end-to-end cells first.")
else:
    zip_path = OUTPUT_DIR / "reddit_test_phase2_end_to_end_outputs.zip"
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for p in existing_output_files:
            if p == zip_path:
                continue
            zf.write(p, arcname=p.relative_to(OUTPUT_DIR) if p.is_relative_to(OUTPUT_DIR) else p.name)
    print(f"Saved zip: {zip_path} | {zip_path.stat().st_size:,} bytes")

    if DRIVE_OUTPUT_AVAILABLE:
        print("Persistent Drive copy is available here:")
        print(zip_path)
    else:
        print("WARNING: Drive output is not available. Download the zip before the Colab runtime disconnects.")

    try:
        from google.colab import files
        files.download(str(zip_path))
    except Exception as exc:
        print(f"Automatic browser download was not started: {exc}")
        print("If needed, use the Colab file browser or run files.download(str(zip_path)) manually.")


